In [1]:
# Load training data and the selected 108-feature configuration.

import json
import pandas as pd
from pathlib import Path
from feature_engineering import create_engineered_features

PROJECT_ROOT = Path.cwd().parent
SPLIT_DIR = PROJECT_ROOT / "fine_tuning" / "data_splits"
ARTIFACT_DIR = PROJECT_ROOT / "fine_tuning" / "artifacts"

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "validation.csv")

with open(ARTIFACT_DIR / "feature_selection_config.json") as f:
    feature_config = json.load(f)

selected_features = feature_config["reduced_features"]

X_train = create_engineered_features(
    train_df.drop(columns=["TARGET", "SK_ID_CURR"])
)[selected_features]
y_train = train_df["TARGET"]

X_val = create_engineered_features(
    val_df.drop(columns=["TARGET", "SK_ID_CURR"])
)[selected_features]
y_val = val_df["TARGET"]

print("Training:", X_train.shape, "| Validation:", X_val.shape)

Training: (215257, 108) | Validation: (46127, 108)


In [2]:
# Cell 2: Preprocess the selected features for XGBoost training.

import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool", "str"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median", add_indicator=True), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore", dtype=np.float32
            ))
        ]), categorical_features)
    ],
    sparse_threshold=1.0
)

X_train_processed = preprocessor.fit_transform(X_train).astype(np.float32)
X_val_processed = preprocessor.transform(X_val).astype(np.float32)

print("Processed:", X_train_processed.shape, "|", X_val_processed.shape)

Processed: (215257, 296) | (46127, 296)


In [3]:
# Cell 3: Compare class-weight strategies using validation performance.

from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

imbalance_ratio = (y_train == 0).sum() / (y_train == 1).sum()

weight_strategies = {
    "unweighted": 1,
    "moderate": np.sqrt(imbalance_ratio),
    "balanced": imbalance_ratio
}

weight_results = []

for name, weight in weight_strategies.items():
    model = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=5,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        scale_pos_weight=weight,
        early_stopping_rounds=50,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_processed, y_train,
        eval_set=[(X_val_processed, y_val)],
        verbose=False
    )

    probabilities = model.predict_proba(X_val_processed)[:, 1]

    weight_results.append({
        "strategy": name,
        "scale_pos_weight": round(weight, 3),
        "best_iteration": model.best_iteration,
        "roc_auc": roc_auc_score(y_val, probabilities),
        "average_precision": average_precision_score(y_val, probabilities)
    })

weight_results_df = pd.DataFrame(weight_results)
display(weight_results_df.sort_values("average_precision", ascending=False))

,strategy,scale_pos_weight,best_iteration,roc_auc,average_precision
1,moderate,3.375,467,0.766389,0.257949
2,balanced,11.387,350,0.765089,0.256333
0,unweighted,1.000,345,0.766007,0.255758


In [4]:
# Cell 4: Tune XGBoost tree depth and regularization.
# Uses moderate class weighting and validation early stopping.

tuning_configs = {
    "baseline": {"max_depth": 5, "min_child_weight": 5},
    "shallower": {"max_depth": 4, "min_child_weight": 5},
    "deeper": {"max_depth": 6, "min_child_weight": 5},
    "stronger_regularization": {"max_depth": 5, "min_child_weight": 10}
}

tuning_results = []
tuned_models = {}

for name, config in tuning_configs.items():
    model = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        scale_pos_weight=3.375,
        early_stopping_rounds=50,
        random_state=42,
        n_jobs=-1,
        **config
    )

    model.fit(X_train_processed, y_train,
              eval_set=[(X_val_processed, y_val)], verbose=False)

    probabilities = model.predict_proba(X_val_processed)[:, 1]
    tuned_models[name] = model

    tuning_results.append({
        "configuration": name,
        **config,
        "best_iteration": model.best_iteration,
        "roc_auc": roc_auc_score(y_val, probabilities),
        "average_precision": average_precision_score(y_val, probabilities)
    })

tuning_results_df = pd.DataFrame(tuning_results).sort_values(
    "average_precision", ascending=False
)

best_name = tuning_results_df.iloc[0]["configuration"]
best_model = tuned_models[best_name]

display(tuning_results_df)

,configuration,max_depth,min_child_weight,best_iteration,roc_auc,average_precision
3,stronger_regularization,5,10,424,0.766218,0.258794
0,baseline,5,5,432,0.766083,0.257923
2,deeper,6,5,323,0.766077,0.257618
1,shallower,4,5,419,0.765611,0.256132


LIGHTGBM

In [5]:
# Cell 5: Benchmark LightGBM against the selected XGBoost model.
# Keeps the dataset, preprocessing and class weighting consistent.

import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

lgb_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    scale_pos_weight=3.375,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_model.fit(
    X_train_processed,
    y_train,
    eval_set=[(X_val_processed, y_val)],
    eval_metric="average_precision",
    callbacks=[lgb.early_stopping(50, verbose=False)]
)

# Evaluate both models with the same sklearn metrics
xgb_prob = best_model.predict_proba(X_val_processed)[:, 1]
lgb_prob = lgb_model.predict_proba(X_val_processed)[:, 1]

comparison = pd.DataFrame([
    {
        "model": "XGBoost",
        "best_iteration": best_model.best_iteration,
        "roc_auc": roc_auc_score(y_val, xgb_prob),
        "average_precision": average_precision_score(y_val, xgb_prob)
    },
    {
        "model": "LightGBM",
        "best_iteration": lgb_model.best_iteration_,
        "roc_auc": roc_auc_score(y_val, lgb_prob),
        "average_precision": average_precision_score(y_val, lgb_prob)
    }
])

display(comparison)

c:\CODING\Credit_Risk\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


,model,best_iteration,roc_auc,average_precision
0,XGBoost,424,0.766218,0.258794
1,LightGBM,9,0.736492,0.217565


In [6]:
# Cell 6: Train CatBoost using native categorical features.
# Uses the same 108 selected features and train/validation partitions.

from catboost import CatBoostClassifier

X_train_cb = X_train.copy()
X_val_cb = X_val.copy()

for col in categorical_features:
    X_train_cb[col] = X_train_cb[col].fillna("__MISSING__").astype(str)
    X_val_cb[col] = X_val_cb[col].fillna("__MISSING__").astype(str)

cat_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    eval_metric="PRAUC",
    scale_pos_weight=3.375,
    l2_leaf_reg=5,
    random_seed=42,
    early_stopping_rounds=100,
    verbose=False,
    allow_writing_files=False
)

cat_model.fit(
    X_train_cb,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_val_cb, y_val),
    verbose=False
)

cat_prob = cat_model.predict_proba(X_val_cb)[:, 1]

comparison = pd.concat([
    comparison,
    pd.DataFrame([{
        "model": "CatBoost",
        "best_iteration": cat_model.get_best_iteration(),
        "roc_auc": roc_auc_score(y_val, cat_prob),
        "average_precision": average_precision_score(y_val, cat_prob)
    }])
], ignore_index=True)

display(comparison)

,model,best_iteration,roc_auc,average_precision
0,XGBoost,424,0.766218,0.258794
1,LightGBM,9,0.736492,0.217565
2,CatBoost,1059,0.767280,0.256744
